# 06 · Demo: baseline vs. condition-aware search (Gradio)

**Purpose.** An interactive interface for the live demo and the user study: one query box, skill and day selectors, and the **baseline and improved rankings side by side**, with explanations, highlighted matching terms, per-query latency, and a switch between **live data** and a **saved snapshot**.

**Inputs:** the search functions from `04_search` and the live-collection function from `00_data_collection` (both loaded with `%run`), snapshots in `data/snapshots/`.

**Outputs:** a local web app (URL printed below). Open it in a browser and press F11 for full-screen presentation.

**Presentation tips**
- Rehearse with *Saved snapshot* on the snapshot you will present. If the network or the APIs fail on the day, the demo is identical.
- *Live data* falls back to the latest snapshot automatically (with a visible notice) when an API is down.
- For the user study, hide the side-by-side view and show one system at a time, labelled neutrally (see `SHOW_SYSTEM_NAMES`).

In [17]:
# ---- Configuration ----
import os
from pathlib import Path


def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "requirements.txt").is_file():
            return folder
    raise FileNotFoundError("Project root not found. Open the project folder in VS Code (File > Open Folder).")


ROOT = find_project_root()
os.chdir(ROOT / "notebooks")
os.environ["SURF_IR_LIBRARY_MODE"] = "1"
%run ./04_search.ipynb
%run ./00_data_collection.ipynb
# Settings below are applied after %run so they take precedence.

DEMO_SNAPSHOT_DATE = None        # None = latest snapshot; set a date to rehearse/present a fixed day
DEFAULT_K = 5
SERVER_NAME = "127.0.0.1"        # local only; nothing is exposed to the internet
SERVER_PORT = None               # None = first free port from 7860
SHARE = False                    # never create a public Gradio link for the presentation
LIVE_CACHE_MINUTES = 30          # reuse a live fetch for this long (avoids hammering the free APIs)
SHOW_SYSTEM_NAMES = True         # False = label columns "System A" / "System B" (user study)
MAP_FILE = ROOT / "results" / "figures" / "surf_spot_map.html"   # created by 07_spot_map.ipynb
MAP_URL = "/gradio_api/file=" + MAP_FILE.as_posix()               # served by the demo server (see allowed_paths)
SAMPLE_N = None                  # not used here
SEED = 42

Library mode: functions defined, nothing fetched.


## 1. Data source: live or saved snapshot
Live mode calls the same `collect_all()` used for daily snapshots (notebook 00), but keeps the result in memory. If the fetch fails (API down, no internet), we fall back to the latest saved snapshot and say so in the interface. A demo must never show an error page.

In [18]:
import gradio as gr

_LIVE = {"snapshot": None, "fetched": None}


def get_live_snapshot():
    """Returns (snapshot, notice). Live data is cached for LIVE_CACHE_MINUTES."""
    now = pd.Timestamp.now()
    if _LIVE["snapshot"] is not None and now - _LIVE["fetched"] < pd.Timedelta(minutes=LIVE_CACHE_MINUTES):
        return _LIVE["snapshot"], None
    try:
        data = collect_all()
        if data["meta"]["status"] == "failed":
            raise RuntimeError("no marine forecast could be fetched")
        snap = snapshot_from_frames(data["marine"], data["wind"], data["obs"], data["meta"],
                                    key=f"live-{data['meta']['fetched_at']}")
        snap["ref_time"] = pd.Timestamp(data["meta"]["fetched_at"])
        _LIVE.update(snapshot=snap, fetched=now)
        return snap, None
    except Exception as exc:
        fallback = load_snapshot(DEMO_SNAPSHOT_DATE)
        return fallback, f"Live data unavailable ({exc}). Showing saved snapshot {fallback['date']}."


def pick_snapshot(source, snapshot_date):
    if source == "Live data":
        return get_live_snapshot()
    return load_snapshot(snapshot_date or DEMO_SNAPSHOT_DATE), None

## 2. Rendering results
Each result card shows: the spot, the verdict for the user's level, the explanation (improved system only), a safety warning if any, and the profile text with the **matched terms highlighted**. The highlighting is computed from the stems that actually matched, so the audience can see query expansion at work (e.g. *newbie* highlighting *beginner*).

In [19]:
CSS = """
.card {border: 1px solid #d0d7de; border-radius: 8px; padding: 10px 12px; margin-bottom: 8px; background: #fff; color: #1f2328;}
.card h4 {margin: 0 0 4px 0;}
.meta {color: #57606a; font-size: 0.85em;}
.badge {display: inline-block; padding: 1px 8px; border-radius: 10px; font-size: 0.8em; font-weight: 600; margin-left: 6px;}
.Good {background: #dafbe1; color: #116329;} .Fair {background: #fff8c5; color: #7d4e00;}
.Poor {background: #ffebe9; color: #a40e26;} .Nodata {background: #eaeef2; color: #57606a;}
.warn {color: #a40e26; font-weight: 600;}
.expl {margin: 6px 0;}
.profile {font-size: 0.85em; color: #424a53;}
mark {background: #fff3a3; padding: 0 1px;}
.outlook td {padding: 2px 6px; text-align: center; font-size: 0.85em;}
"""

DAY_CHOICES = ["From query", "Today", "Tomorrow", "Day after tomorrow"]
SKILL_CHOICES = ["From query", "beginner", "intermediate", "advanced"]


def render_cards(info, show_conditions):
    if not info["results"]:
        return "<p>No results.</p>"
    stems = set(analyze(info["parsed"].text)) | {s for exp in info["expansions"] for s in analyze(exp)}
    html = []
    for r in info["results"]:
        spot = SPOTS[r["spot_id"]]
        badge = f"<span class='badge {r['verdict'].replace(' ', '')}'>{escape(r['verdict'])}</span>" if show_conditions and r["verdict"] else ""
        dist = f" · {r['distance_km']:.0f} km away" if r["distance_km"] is not None else ""
        body = (f"<div class='expl'>{escape(r['explanation'])}</div>" if show_conditions and r["explanation"]
                else "<div class='expl meta'>Text match only (no condition information).</div>")
        warn = f"<div class='warn'>⚠ {escape(r['warning'])}</div>" if show_conditions and r["warning"] else ""
        matched = ", ".join(r["matched_stems"]) or "none"
        map_link = (f" · <a href='{MAP_URL}#{r['spot_id']}' target='_blank'>📍 Show on map</a>" if MAP_FILE.exists() else "")
        html.append(
            f"<div class='card'><h4>{r['rank']}. {escape(r['name'])}{badge}</h4>"
            f"<div class='meta'>{escape(spot['region'])} · {escape(spot['break_type'])} break{dist} · score {r['score']:.2f}{map_link}</div>"
            f"{body}{warn}<div class='profile'>{highlight(spot['description'], stems)}</div>"
            f"<div class='meta'>Matched terms: {escape(matched)}</div></div>")
    return "".join(html)


def render_outlook(results, snap, skill):
    """7-day timeline: best condition-fit score per day for the top 3 improved results."""
    if not results:
        return ""
    rows = {r["name"]: daily_outlook(r["spot_id"], snap, skill) for r in results[:3]}
    days = sorted({d for s in rows.values() for d in s.index})
    head = "".join(f"<th>{pd.Timestamp(d):%a %d}</th>" for d in days)
    body = ""
    for name, s in rows.items():
        cells = ""
        for d in days:
            v = s.get(d, np.nan)
            color = "#eaeef2" if pd.isna(v) else ("#dafbe1" if v >= THRESHOLDS[skill]["good_threshold"]
                                                  else "#fff8c5" if v >= THRESHOLDS[skill]["surfable_threshold"] else "#ffebe9")
            cells += f"<td style='background:{color}'>{'' if pd.isna(v) else f'{v:.2f}'}</td>"
        body += f"<tr><td style='text-align:left'>{escape(name)}</td>{cells}</tr>"
    return (f"<b>7-day outlook ({escape(skill)}, best condition-fit score per day)</b>"
            f"<table class='outlook'><tr><th></th>{head}</tr>{body}</table>")

## 3. The search callback
Both columns call the same `search_verbose()` used by the evaluation, on the same snapshot and with the same overrides, so what the audience sees is exactly what was evaluated.

In [20]:
def run_search(query, skill_choice, day_choice, source, snapshot_date, k):
    if not query or not query.strip():
        return "Please type a query.", "", "", ""
    snap, notice = pick_snapshot(source, snapshot_date)
    ref = snap["ref_time"]
    skill = None if skill_choice == "From query" else skill_choice
    day = None if day_choice == "From query" else (ref.normalize() + pd.Timedelta(days=DAY_CHOICES.index(day_choice) - 1)).date().isoformat()
    base = search_verbose(query, "baseline", int(k), snapshot=snap, skill=skill, day=day)
    imp = search_verbose(query, "improved", int(k), snapshot=snap, skill=skill, day=day)
    pq = imp["parsed"]
    source_line = (f"Live data fetched {ref:%Y-%m-%d %H:%M} (Taiwan time)" if snap["key"].startswith("live")
                   else f"Saved snapshot {snap['date']} (fetched {snap['meta']['fetched_at'].replace('T', ' ')} Taiwan time)")
    info = [f"**Understood:** {describe_parsed(pq)}"]
    if imp["expansions"]:
        info.append(f"**Jargon expansion:** {', '.join(sorted(set(imp['expansions'])))}")
    info += [f"**Notice:** {n}" for n in pq.notes + ([notice] if notice else [])]
    info.append(f"**Data:** {source_line}")
    label_b, label_i = ("Baseline (BM25)", "Improved (condition-aware)") if SHOW_SYSTEM_NAMES else ("System A", "System B")
    head_b = f"<h3>{label_b}</h3><div class='meta'>{base['latency_ms']['total']:.0f} ms</div>"
    head_i = (f"<h3>{label_i}</h3><div class='meta'>{imp['latency_ms']['total']:.0f} ms "
              f"(understanding {imp['latency_ms']['understanding']:.0f} · text {imp['latency_ms']['text']:.0f} · "
              f"conditions {imp['latency_ms']['conditions']:.0f})</div>")
    return ("  \n".join(info), head_b + render_cards(base, show_conditions=False),
            head_i + render_cards(imp, show_conditions=True), render_outlook(imp["results"], snap, pq.skill))

## 4. The interface
Required data attribution (DWD / Open-Meteo / CWA) is always visible at the bottom of the page. The **Open surf spot map** button (top right) opens the interactive map from `07_spot_map` in a new tab, and every result card has a **Show on map** link that opens the map zoomed to that spot. If the map file does not exist yet, run `07_spot_map.ipynb` once.

In [21]:
ATTRIBUTION = (
    "**Data sources.** Wave, swell, sea-level and wind forecasts: [Open-Meteo.com](https://open-meteo.com/) "
    "(licensed CC BY 4.0), including wave model data from the Deutscher Wetterdienst (DWD). "
    "Buoy and tide observations: Central Weather Administration (CWA) Open Data, Taiwan. "
    "Spot profiles: written by the project team from cited public sources. "
    "Forecasts describe offshore conditions; always check conditions on the beach. Educational project (MIS590), non-commercial."
)


MAP_BUTTON = (
    f"<div style='text-align:right; padding-top:18px'><a href='{MAP_URL}' target='_blank' "
    "style='display:inline-block; padding:10px 18px; border-radius:8px; background:#1f6feb; color:white; "
    "font-weight:600; text-decoration:none'>🗺️ Open surf spot map</a></div>"
    if MAP_FILE.exists() else
    "<div style='text-align:right; padding-top:18px; color:#57606a'>🗺️ Map not found: run 07_spot_map.ipynb once</div>"
)


def build_demo():
    snapshots = list_snapshots()
    with gr.Blocks(title="Surf Spot Search Taiwan", css=CSS, theme=gr.themes.Soft()) as app:
        with gr.Row():
            gr.Markdown("# 🏄 Surf Spot Search Taiwan\nAsk in plain English, e.g. *beginner-friendly beach break near Kaohsiung this weekend*.")
            gr.HTML(MAP_BUTTON)
        with gr.Row():
            query = gr.Textbox(label="Query", placeholder="uncrowded reef break on the east coast tomorrow morning", scale=4)
            search_btn = gr.Button("Search", variant="primary", scale=1)
        with gr.Row():
            skill = gr.Dropdown(SKILL_CHOICES, value="From query", label="Skill level")
            day = gr.Dropdown(DAY_CHOICES, value="From query", label="Day")
            source = gr.Radio(["Saved snapshot", "Live data"], value="Saved snapshot", label="Data")
            snapshot = gr.Dropdown(snapshots, value=DEMO_SNAPSHOT_DATE or (snapshots[-1] if snapshots else None), label="Snapshot")
            k = gr.Slider(1, 10, value=DEFAULT_K, step=1, label="Results")
        gr.Examples(EXAMPLE_QUERIES, inputs=[query])
        info = gr.Markdown()
        with gr.Row():
            baseline_col = gr.HTML()
            improved_col = gr.HTML()
        outlook = gr.HTML()
        gr.Markdown(ATTRIBUTION)
        inputs = [query, skill, day, source, snapshot, k]
        outputs = [info, baseline_col, improved_col, outlook]
        search_btn.click(run_search, inputs, outputs)
        query.submit(run_search, inputs, outputs)
    return app

## 5. Self-check before launching
Runs one example through the exact callback the interface uses. If this cell fails, the demo would fail too, so you find out before the audience does.

In [22]:
_check = run_search(EXAMPLE_QUERIES[0], "From query", "From query", "Saved snapshot", DEMO_SNAPSHOT_DATE, 3)
print(_check[0].replace("**", ""))
print("Baseline and improved columns rendered:", all(len(x) > 0 for x in _check[1:3]))

Understood: Location: Kaohsiung · Skill: beginner · Time: this weekend  
Data: Saved snapshot 2026-10-16 (fetched 2026-10-16 11:21:03 Taiwan time)
Baseline and improved columns rendered: True


## 6. Launch
The app appears inline below; the printed URL opens the same app in a browser (use that one for the presentation, full screen with F11). Re-running this cell restarts the server.

In [23]:
if "demo" in globals():
    try:
        demo.close()
    except Exception:
        pass
demo = build_demo()
demo.launch(server_name=SERVER_NAME, server_port=SERVER_PORT, share=SHARE, inline=True,
            prevent_thread_lock=True, quiet=True, allowed_paths=[str(MAP_FILE.parent)])
print(f"Demo running at {demo.local_url}  (open in a browser for full screen)")

Closing server running on port: 7860


Demo running at http://127.0.0.1:7860/  (open in a browser for full screen)


In [ ]:
STOP_DEMO = False      # set to True and run this cell to stop the server
if STOP_DEMO:
    demo.close()

08:23:56 WARNING cwa[stations=O-B0076-001]: SSLError: HTTPSConnectionPool(host='opendata.cwa.gov.tw', port=443): Max retries exceeded with url: /fileapi/v1/opendataapi/O-B0076-001?Authorization=***&downloadType=WEB&format=JSON (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Missing Subject Key Identifier (_ssl.c:1032)')))
08:24:18 WARNING cwa[obs_48h=O-B0075-001]: SSLError: HTTPSConnectionPool(host='opendata.cwa.gov.tw', port=443): Max retries exceeded with url: /api/v1/rest/datastore/O-B0075-001?Authorization=***&format=JSON (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Missing Subject Key Identifier (_ssl.c:1032)')))
